# v5 dash-cam pothole model — issue #118 T4 (train) + T5 (score)

**Ship rule:** Detroit exam box F1 **> 0.190** AND crack false boxes **≤ the shipped model's** (17% on Ayaan's set).
Shipped now (#117): 70/30 blend of (v3+v1) and (v4+v1 85/15), thr 0.40.

**Data (all from `main`, nothing in git):**
1. Detroit reviewed labels (1,273 frames, 427 with potholes incl. boxed city photos) → `build_v4_data.py --conf 0.35`
2. `rdd_d40_clean.py` — clean RDD2022 Japan + India D40 potholes (5,430 boxes), **capped ≈2× Detroit**
3. `city_paste.py` — boxed Detroit city photos pasted small/distant onto empty Detroit road (~600 imgs)
4. RDD2022 **US crack-only** photos as "not a pothole" (≈1× Detroit); RDD US **test** split held out for the crack check
5. `style_aug.py` footage-style copies (grey / wet / glare / haze / blur / JPEG) of the Detroit + city-paste images
**Train** from `v4_best.pt`, 640, style_aug's recommended hyperparams, **val = Detroit only**.
**Then** blends: v5+v1 (0.70 / 0.85) and v5 × shipped (0.70) → Detroit exam + crack check → auto verdict.

**Setup:** A100 · 🔑 Secrets: `HF_TOKEN` (strongly recommended — HF throttles anonymous downloads) · `Runtime → Run all`.
~60–80 min, ~12–16 units. Hard stop for model work ~6 AM ET (freeze 8 AM).

## 1 · GPU, Drive, repo, starting weights

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU"
!pip -q install ultralytics onnx onnxruntime onnxslim huggingface_hub
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil, random, json, yaml
try:
    from google.colab import userdata
    _t = userdata.get('HF_TOKEN')
    if isinstance(_t, dict): _t = _t.get('value')
    if _t: os.environ['HF_TOKEN'] = _t; print('HF_TOKEN set')
except Exception as e:
    print('no HF_TOKEN secret (downloads will be slower):', type(e).__name__)
DRIVE_DIR = '/content/drive/MyDrive/still-there-potholes'
if os.path.isdir('/content/repo/.git'):
    !git -C /content/repo pull -q
else:
    !git clone -q https://github.com/ayaanamir/RoadMapWayneAWSHackathon /content/repo
AL = '/content/repo/pipeline/autolabel'
V4 = f'{DRIVE_DIR}/handoff_v4/v4_best.pt'; assert os.path.exists(V4), 'v4_best.pt missing — run train_v4.ipynb first'
shutil.copy(V4, '/content/v4_best.pt')
V1_ONNX = '/content/repo/pipeline/train/results/v1_phone/pothole.onnx'
SHIPPED = '/content/repo/apps/live/model/pothole-dashcam.onnx'          # #117 blend, F1 0.190
for p in ('/content/v4_best.pt', V1_ONNX, SHIPPED): print(os.path.exists(p), p)

## 2 · Detroit labels → Detroit v4 set (mistake mining with v4)

In [ ]:
!python3 {AL}/pull_reviewed.py --api https://wayneawshackathon.vercel.app --out /content/detroit
!python3 {AL}/build_v4_data.py --images /content/detroit/train/images --labels /content/detroit/train/labels --model /content/v4_best.pt --conf 0.35 --out /content/ds_v4
def count(d): return len(glob.glob(f'{d}/*.jpg')) + len(glob.glob(f'{d}/*.png'))
N_DET = count('/content/ds_v4/images/train')
print('Detroit v4 train images:', N_DET, '| val:', count('/content/ds_v4/images/val'),
      '| exam (never trained):', count('/content/detroit/test/images'))

## 3 · Clean RDD2022 Japan + India D40 potholes (remote zip reads, ~5–10 min)

In [ ]:
if not glob.glob('/content/rdd_d40/images/*.jpg'):
    !python3 {AL}/rdd_d40_clean.py --out /content/rdd_d40 --countries Japan India
print(open('/content/rdd_d40/summary.json').read()[:600])

## 4 · City photos → dash-cam pastes (src = boxed `crowd_scf-*`, dest = empty Detroit train frames)

In [ ]:
def link_set(pairs, root):
    shutil.rmtree(root, ignore_errors=True)
    for d in ('images', 'labels'): os.makedirs(f'{root}/{d}', exist_ok=True)
    for img, lab in pairs:
        os.symlink(img, f'{root}/images/{os.path.basename(img)}')
        os.symlink(lab, f'{root}/labels/{os.path.basename(lab)}')
    return len(pairs)
det_tr = sorted(glob.glob('/content/detroit/train/images/*.jpg'))
lab_of = lambda p: p.replace('/images/', '/labels/')[:-4] + '.txt'
scf = [(p, lab_of(p)) for p in det_tr if os.path.basename(p).startswith('crowd_scf-')
       and os.path.exists(lab_of(p)) and os.path.getsize(lab_of(p)) > 0]
empty = [(p, lab_of(p)) for p in det_tr if not os.path.basename(p).startswith('crowd_scf-')
         and os.path.exists(lab_of(p)) and os.path.getsize(lab_of(p)) == 0]
print('boxed city photos:', link_set(scf, '/content/scf_src'), '| empty Detroit dest frames:', link_set(empty, '/content/scf_dest'))
if not glob.glob('/content/city_paste/images/*.jpg'):
    !python3 {AL}/city_paste.py --src-images /content/scf_src/images --src-labels /content/scf_src/labels --dest-images /content/scf_dest/images --dest-labels /content/scf_dest/labels --out /content/city_paste --n-out 600
print('city_paste images:', count('/content/city_paste/images'))

## 5 · RDD2022 US crack-only negatives (train) + held-out crack check (test split)

In [ ]:
CRACK_CODE = True  # marker for the offline test
from huggingface_hub import snapshot_download
def build_us_cracks(src='/content/rdd_us_raw', neg_out='/content/us_neg', exam_out='/content/crack_exam', n_neg=1000, seed=0):
    random.seed(seed)
    for d in ('images', 'labels'): os.makedirs(f'{neg_out}/{d}', exist_ok=True)
    os.makedirs(exam_out, exist_ok=True)
    train_cracks, exam = [], []
    for lab in sorted(glob.glob(f'{src}/data/labels/*/*/United_States_*.txt')):
        rows = [l.split() for l in open(lab) if l.strip()]
        if not rows: continue                                   # need visible damage to be a useful negative
        img = lab.replace('/labels/', '/images/')[:-4] + '.jpg'
        if not os.path.exists(img): continue
        (exam if '/test/' in lab.replace('\\', '/') else train_cracks).append(img)
    for img in random.sample(train_cracks, min(n_neg, len(train_cracks))):
        name = 'usneg_' + os.path.basename(img)[:-4]
        shutil.copy(img, f'{neg_out}/images/{name}.jpg'); open(f'{neg_out}/labels/{name}.txt', 'w').close()
    for img in exam: shutil.copy(img, exam_out)
    print('US crack photos: train pool', len(train_cracks), '→ negatives', count(f'{neg_out}/images'), '| crack exam', len(exam))

if not glob.glob('/content/us_neg/images/*.jpg'):
    snapshot_download('dronefreak/RDD2022', repo_type='dataset', local_dir='/content/rdd_us_raw',
                      allow_patterns=['data/images/*/*/United_States_*', 'data/labels/*/*/United_States_*.txt'], max_workers=16)
    build_us_cracks(n_neg=N_DET)                                # ≈1× Detroit

## 6 · Mix with caps + footage-style augmentation, one yaml (val = Detroit only)

In [ ]:
random.seed(1)
def capped(src_root, n, root):
    imgs = sorted(glob.glob(f'{src_root}/images/*.jpg'))
    pick = random.sample(imgs, min(n, len(imgs)))
    return link_set([(p, lab_of(p)) for p in pick], root)
n_rdd = capped('/content/rdd_d40', 2 * N_DET, '/content/mix/rdd_d40')          # cap ≈2× Detroit
print('RDD D40 used:', n_rdd)
if not glob.glob('/content/style_aug/images/*.jpg'):
    !python3 {AL}/style_aug.py --pair /content/ds_v4/images/train /content/ds_v4/labels/train --pair /content/city_paste/images /content/city_paste/labels --out /content/style_aug --frac 0.5
REC = json.load(open('/content/style_aug/summary.json')).get('recommended_ultralytics', {})
TRAIN_DIRS = ['/content/ds_v4/images/train', '/content/mix/rdd_d40/images', '/content/city_paste/images',
              '/content/us_neg/images', '/content/style_aug/images']
for d in glob.glob('/content/**/labels*.cache', recursive=True): os.remove(d)
yaml.safe_dump({'train': TRAIN_DIRS, 'val': '/content/ds_v4/images/val', 'nc': 1, 'names': ['pothole']},
               open('/content/v5.yaml', 'w'))
for d in TRAIN_DIRS: print(f'{count(d):6d}  {d}')
print('total train:', sum(count(d) for d in TRAIN_DIRS), '| val (Detroit):', count('/content/ds_v4/images/val'))
print('style_aug recommended:', REC)

## 7 · Train v5 from v4 (≈30–45 min on A100) — resumes after a disconnect

In [ ]:
from ultralytics import YOLO
RUN = f'{DRIVE_DIR}/runs/v5'
last, done = f'{RUN}/weights/last.pt', f'{RUN}/weights/TRAINING_DONE'
hp = dict(imgsz=640, hsv_h=0.008, hsv_s=0.4, hsv_v=0.25, degrees=0.0, translate=0.05, scale=0.15, fliplr=0.5,
          mosaic=1.0, close_mosaic=10, erasing=0.0, mixup=0.0, copy_paste=0.0)
hp.update({k: v for k, v in REC.items() if k in hp})
if os.path.exists(done):
    print('Already trained — skipping. Delete', done, 'to train again.')
elif os.path.exists(last):
    print('Resuming', last); YOLO(last).train(resume=True)
else:
    YOLO('/content/v4_best.pt').train(data='/content/v5.yaml', epochs=40, patience=12, lr0=0.002, warmup_epochs=1,
        batch=-1, cache='ram', workers=min(8, os.cpu_count() or 2),
        project=f'{DRIVE_DIR}/runs', name='v5', exist_ok=True, **hp)
open(done, 'w').close()
V5_PT = f'{RUN}/weights/best.pt'

## 8 · Export + blends: v5+v1 (0.70 / 0.85), v5 × shipped (0.70)

In [ ]:
import onnx, numpy as np
from onnx import numpy_helper
V5 = YOLO(V5_PT).export(format='onnx', imgsz=640)
def blend(a_path, b_path, a, out):
    A = onnx.load(a_path); B = {i.name: numpy_helper.to_array(i) for i in onnx.load(b_path).graph.initializer}
    n = 0
    for k, init in enumerate(A.graph.initializer):
        w = numpy_helper.to_array(init)
        if w.dtype == np.float32 and init.name in B and B[init.name].shape == w.shape:
            A.graph.initializer[k].CopyFrom(numpy_helper.from_array((a*w + (1-a)*B[init.name]).astype(np.float32), init.name)); n += 1
    onnx.save(A, out); print(f'{os.path.basename(out)}: blended {n}/{len(A.graph.initializer)} tensors')
os.makedirs('/content/exam_models', exist_ok=True)
named = {'v5': '/content/exam_models/v5.onnx'}; shutil.copy(V5, named['v5'])
for a in (0.7, 0.85):
    named[f'v5v1_{int(a*100)}'] = f'/content/exam_models/v5v1_{int(a*100)}.onnx'; blend(V5, V1_ONNX, a, named[f'v5v1_{int(a*100)}'])
named['v5xshipped_70'] = '/content/exam_models/v5xshipped_70.onnx'; blend(V5, SHIPPED, 0.7, named['v5xshipped_70'])
named['shipped'] = '/content/exam_models/shipped.onnx'; shutil.copy(SHIPPED, named['shipped'])

## 9 · T5 — Detroit exam (215 frames) + crack check + automatic verdict

In [ ]:
args = ' '.join(f'--model {p}' for p in named.values())
!python3 {AL}/eval_models.py --images /content/detroit/test/images --labels /content/detroit/test/labels --out /content/eval_v5 --modes 640 --device 0 {args}
REPORT = open('/content/eval_v5/report.md').read(); print(REPORT)
import re
F1, THR = {}, {}
for line in REPORT.replace('\\', '').splitlines():           # tolerate markdown-escaped underscores
    m = re.match(r'\|\s*\d+\s*\|\s*([\w.]+?)\.onnx \[640\]\s*\|\s*([\d.]+)\s*\|\s*([\d.]+)', line)
    if m: F1[m.group(1)], THR[m.group(1)] = float(m.group(2)), float(m.group(3))
CRACKS = sorted(glob.glob('/content/crack_exam/*.jpg')); CR = {}
for name, p in named.items():
    model, hit = YOLO(p, task='detect'), 0
    t = THR.get(name, 0.40)
    for i in range(0, len(CRACKS), 32):
        for r in model.predict(CRACKS[i:i+32], conf=t, imgsz=640, verbose=False): hit += len(r.boxes) > 0
    CR[name] = hit / max(1, len(CRACKS))
base_f1, base_cr = F1.get('shipped', 0.190), CR.get('shipped', 0.17)
rows = ['| model | Detroit box F1 | thr | crack photos wrongly boxed (at thr) | passes ship rule |', '|---|---|---|---|---|']
for name in named:
    ok = name != 'shipped' and F1.get(name, 0) > max(0.190, base_f1) and CR[name] <= base_cr
    rows.append(f"| {name} | {F1.get(name, float('nan')):.3f} | {THR.get(name, 0.40):.2f} | {CR[name]:.0%} | {'✅' if ok else '—'} |")
winners = [n for n in named if n != 'shipped' and F1.get(n, 0) > max(0.190, base_f1) and CR[n] <= base_cr]
BEST = max(winners, key=lambda n: F1[n]) if winners else None
SUMMARY = '\n'.join(rows) + f"\n\nCrack check: {len(CRACKS)} held-out RDD US crack-only photos, each model at its own best-F1 threshold.\n" + \
          (f'**Verdict: SHIP `{BEST}`** (F1 {F1[BEST]:.3f} > {max(0.190, base_f1):.3f}, cracks {CR[BEST]:.0%} ≤ shipped {base_cr:.0%}), thr {THR[BEST]:.2f}.' if BEST
           else '**Verdict: keep the shipped model** — no v5 variant beats it on both F1 and cracks.')
print(SUMMARY)
os.makedirs(f'{DRIVE_DIR}/handoff_v5', exist_ok=True)
open(f'{DRIVE_DIR}/handoff_v5/v5_report.md', 'w').write(REPORT + '\n\n## T5 summary\n' + SUMMARY + '\n')

## 10 · Save candidates to Drive (`handoff_v5/`), then disconnect

In [ ]:
for name, p in named.items():
    if name != 'shipped': shutil.copy(p, f'{DRIVE_DIR}/handoff_v5/{name}.onnx')
shutil.copy(V5_PT, f'{DRIVE_DIR}/handoff_v5/v5_best.pt')
for f in sorted(os.listdir(f'{DRIVE_DIR}/handoff_v5')): print(f'{f:24s} {os.path.getsize(f"{DRIVE_DIR}/handoff_v5/{f}")/1e6:.1f} MB')
print('\nDONE — Runtime → Disconnect and delete runtime to stop burning units.')